# GARCH(1,1): the Fiorentini-Calzolari-Panattoni benchmark

Does Julia's GARCH(1,1) give the published answer? Fiorentini, Calzolari and Panattoni (1996) computed the maximum likelihood estimates on `bg96`, the daily Deutsche mark / British pound returns of Bollerslev and Ghysels (1996); Brooks, Burke and Persand (2001, Section 2) quote them to 6 significant figures and found that only one of the nine packages they tested reached them with its default settings. Step 5 fits the model with ARCHModels and prints each value in full, rounded to the published precision, and beside the published one.

Steps 1 to 4 show the model at work first: load a daily series, turn it into daily changes, test whether volatility clusters, and draw volatility bands, from a GARCH model if it does, constant if it doesn't.

Each step is a function, defined in the step's cell and called below it; fold a function to see only its call. To rerun on another series, change the parameters and run all cells.

In [ ]:
Pollis.packages("ARCHModels", "Plots", "PollisDatasets")

In [ ]:
using ARCHModels, Dates, Plots, PollisDatasets, Statistics

## Parameters

In [ ]:
# The dataset (from PollisDatasets) and how to read it
id = "dgs10"          # 10-Year Treasury Yield
value = :yield        # the column with the series
date = :date          # its date column, or nothing
from = Date(2000, 1, 1)  # first day used, or nothing for all
kind = :difference    # :difference (yields), :logreturn (prices), :none (already returns)

## 1. Load the series

In [ ]:
"""
    load_series(id, value; date = nothing, from = nothing)

Load one column of a PollisDatasets dataset as a series, without the missing days.

Needs: a dataset id and the name of its value column (and, if it has one, its
date column). Gives: `series`, a named tuple `(dates, values)`; `dates` is
`nothing` for a dataset without dates.
"""
function load_series(id, value; date = nothing, from = nothing)
	table = dataset(id)
	values = getproperty(table, value)
	dates = date === nothing ? nothing : getproperty(table, date)
	keep = .!ismissing.(values)
	if dates !== nothing && from !== nothing
		keep .&= dates .>= from
	end
	return (dates = dates === nothing ? nothing : dates[keep], values = Float64.(values[keep]))
end

series = load_series(id, value; date = date, from = from)
length(series.values)

## 2. Compute daily changes

In [ ]:
"""
    daily_changes(series; kind = :difference)

Turn levels into daily changes: `:difference` for yields and rates (changes in
basis points), `:logreturn` for prices (percentage log returns), `:none` for a
series that is already returns.

Needs: `series`. Gives: `changes`, a named tuple `(dates, values)` one day
shorter than `series` (the same length for `:none`).
"""
function daily_changes(series; kind = :difference)
	v = series.values
	kind === :none && return series
	values = kind === :difference ? 100 .* diff(v) :
		kind === :logreturn ? 100 .* diff(log.(v)) :
		throw(ArgumentError("kind must be :difference, :logreturn or :none"))
	dates = series.dates === nothing ? nothing : series.dates[2:end]
	return (dates = dates, values = values)
end

changes = daily_changes(series; kind = kind)
length(changes.values)

## 3. Test for ARCH effects

Does volatility cluster? The answer chooses step 4.

In [ ]:
"""
    arch_effects(changes; lags = 5, level = 0.05)

Engle's LM test for ARCH effects: does volatility cluster?

Needs: `changes`. Gives: `found`, true when the test rejects constant variance
at `level`; prints the test.
"""
function arch_effects(changes; lags = 5, level = 0.05)
	test = ARCHLMTest(changes.values .- mean(changes.values), lags)
	display(test)
	return pvalue(test) < level
end

found = arch_effects(changes)

## 4. Volatility bands

If ARCH effects are found, fit a GARCH(1, 1) model and draw its bands; otherwise draw constant bands.

In [ ]:
"""
    garch_fit(changes; p = 1, q = 1)

Fit a GARCH(p, q) model with a constant mean, the code of the GARCH tab of
GARCH-Type Models.

Needs: `changes`. Gives: `model`, the fitted model.
"""
garch_fit(changes; p = 1, q = 1) = fit(GARCH{p, q}, changes.values)

"""
    garch_bands(changes, model)

Plot the changes with bands of plus and minus two conditional standard
deviations from the fitted model.

Needs: `changes`, `model`. Gives: the plot.
"""
garch_bands(changes, model) = bands(changes, volatilities(model), "GARCH volatility bands")

"""
    constant_bands(changes)

Plot the changes with bands of plus and minus two standard deviations, the same
for every day.

Needs: `changes`. Gives: the plot.
"""
constant_bands(changes) = bands(changes, fill(std(changes.values), length(changes.values)), "Constant volatility bands")

"""Draws a series with a band of plus and minus two times `sigma` around zero."""
function bands(changes, sigma, title)
	x = changes.dates === nothing ? eachindex(changes.values) : changes.dates
	p = plot(x, changes.values; label = "daily changes", lw = 0.4, color = :gray, title = title, legend = :topright)
	plot!(p, x, 2 .* sigma; label = "+/- 2 sigma", color = :firebrick, lw = 1)
	plot!(p, x, -2 .* sigma; label = "", color = :firebrick, lw = 1)
	return p
end

if found
	model = garch_fit(changes)
	display(model)
	garch_bands(changes, model)
else
	constant_bands(changes)
end

## 5. Replicate: published results

Before trusting the GARCH step, rerun it on data with published results: `bg96`, the daily Deutsche mark / British pound returns of Bollerslev and Ghysels (1996), on which Fiorentini, Calzolari and Panattoni (1996) computed the benchmark quoted to 6 significant figures in Brooks, Burke and Persand (2001, Section 2). Each value is shown in full, rounded to the published precision, and beside the published one.

The same check is the **Replicate** Next Step of GARCH-Type Models (Next Steps › Compare, GARCH tab); its Learn More tutorial Replicate Published Results explains the gap.

In [ ]:
using Printf

bg = daily_changes(load_series("bg96", :return); kind = :none)
benchmark = garch_fit(bg)
published = [0.0107613, 0.805974, 0.153134, -0.00619041]  # omega, beta, alpha, mu: the order of coef(benchmark)
@printf("%-8s%16s%14s%14s\n", "", "Julia", "rounded", "published")
for (name, e, p) in zip(coefnames(benchmark), coef(benchmark), published)
	digits = something(findlast(n -> round(e, sigdigits=n) == round(p, sigdigits=n), 1:6), 0)  # significant figures in common
	@printf("%-8s%16.10f%14s%14s  %s\n", name, e, string(round(e, sigdigits=6)), string(p), digits == 0 ? "DIFFERENT" : "same to $digits significant figures")
end
# The gap comes from the start-up: ARCHModels sets the first day's variance to the sample variance, the benchmark
# starts the recursion from the mean squared residual (Brooks, Burke and Persand 2001, Section 2, equation 4).